In [ ]:
!pip install -q open_clip_torch transformers scikit-learn matplotlib pillow pandas numpy tqdm gdown pynvml

In [ ]:
import os
import time
import zipfile
import gdown
import torch
import open_clip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from tqdm import tqdm

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    confusion_matrix,

    classification_report,

    roc_curve,

    auc
)

from torch import nn

from torch.utils.data import (

    Dataset,

    DataLoader
)

In [ ]:
# ============================================================
# STEP 3 — DOWNLOAD MULTIOFF DATASET
# ============================================================
!pip install -q gdown

import gdown

folder_url = "https://drive.google.com/drive/folders/1hKLOtpVmF45IoBmJPwojgq6XraLtHmV6"

gdown.download_folder(

    url=folder_url,

    output="MultiOFF",

    quiet=False,

    remaining_ok=True
)

In [ ]:
DATASET_DIR = "./MultiOFF"

TRAIN_FILE = os.path.join(

    DATASET_DIR,

    "Split Dataset",

    "Training_meme_dataset.csv"
)

DEV_FILE = os.path.join(

    DATASET_DIR,

    "Split Dataset",

    "Validation_meme_dataset.csv"
)

TEST_FILE = os.path.join(

    DATASET_DIR,

    "Split Dataset",

    "Testing_meme_dataset.csv"
)

IMAGE_DIR = os.path.join(

    DATASET_DIR,

    "Labelled Images"
)

print(TRAIN_FILE)

print(DEV_FILE)

print(TEST_FILE)

print(IMAGE_DIR)

In [ ]:
train_df = pd.read_csv(
    TRAIN_FILE
)

dev_df = pd.read_csv(
    DEV_FILE
)

test_df = pd.read_csv(
    TEST_FILE
)

In [ ]:
print(train_df.head())

print(train_df.columns)

In [ ]:
train_df["label"] = (

    train_df["label"]

    .astype(str)

    .str.strip()
)

dev_df["label"] = (

    dev_df["label"]

    .astype(str)

    .str.strip()
)

test_df["label"] = (

    test_df["label"]

    .astype(str)

    .str.strip()
)

In [ ]:
label_map = {

    "offensive": 1,

    "Non-offensiv": 0,

    "non-offensiv": 0,

    "non-offensive": 0
}

In [ ]:
train_df["label"] = train_df["label"].map(
    label_map
)

dev_df["label"] = dev_df["label"].map(
    label_map
)

test_df["label"] = test_df["label"].map(
    label_map
)

In [ ]:
train_df = train_df.dropna(
    subset=["label"]
)

dev_df = dev_df.dropna(
    subset=["label"]
)

test_df = test_df.dropna(
    subset=["label"]
)

In [ ]:
train_df["label"] = train_df["label"].astype(int)

dev_df["label"] = dev_df["label"].astype(int)

test_df["label"] = test_df["label"].astype(int)

In [ ]:
train_df = train_df.head(1000)

dev_df = dev_df.head(500)

test_df = test_df.head(500)

In [ ]:
def filter_existing_images(df):

    valid_rows = []

    for idx in range(len(df)):

        row = df.iloc[idx]

        image_path = os.path.join(

            IMAGE_DIR,

            str(row["image_name"]).strip()
        )

        if os.path.exists(image_path):

            valid_rows.append(row)

    return pd.DataFrame(valid_rows)

In [ ]:
train_df = filter_existing_images(
    train_df
)

dev_df = filter_existing_images(
    dev_df
)

test_df = filter_existing_images(
    test_df
)

In [ ]:
print(len(train_df))

print(len(dev_df))

print(len(test_df))

In [ ]:
device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"
)

model, _, preprocess = open_clip.create_model_and_transforms(

    'ViT-B-32',

    pretrained='openai'
)

tokenizer = open_clip.get_tokenizer(
    'ViT-B-32'
)

model = model.to(device)

model.eval()

In [ ]:
class MultiOFFDataset(Dataset):

    def __init__(

        self,

        dataframe
    ):

        self.dataframe = dataframe

    def __len__(self):

        return len(
            self.dataframe
        )

    def __getitem__(

        self,

        idx
    ):

        row = self.dataframe.iloc[idx]

        image_path = os.path.join(

            IMAGE_DIR,

            row["image_name"]
        )

        image = Image.open(

            image_path

        ).convert("RGB")

        image = preprocess(
            image
        )

        text = str(
            row["sentence"]
        )

        label = torch.tensor(

            int(row["label"]),

            dtype=torch.long
        )

        return {

            "image": image,

            "text": text,

            "label": label
        }

In [ ]:
train_dataset = MultiOFFDataset(
    train_df
)

dev_dataset = MultiOFFDataset(
    dev_df
)

test_dataset = MultiOFFDataset(
    test_df
)

In [ ]:
train_loader = DataLoader(

    train_dataset,

    batch_size=16,

    shuffle=True
)

dev_loader = DataLoader(

    dev_dataset,

    batch_size=16,

    shuffle=False
)

test_loader = DataLoader(

    test_dataset,

    batch_size=16,

    shuffle=False
)

In [ ]:
class CLIPClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.clip_model = model

        self.fc = nn.Linear(

            1024,

            2
        )

    def forward(

        self,

        images,

        texts
    ):

        text_tokens = tokenizer(
            texts
        ).to(device)

        with torch.no_grad():

            image_features = self.clip_model.encode_image(
                images
            )

            text_features = self.clip_model.encode_text(
                text_tokens
            )

        combined = torch.cat(

            [

                image_features,

                text_features
            ],

            dim=1
        )

        output = self.fc(
            combined
        )

        return output

In [ ]:
classifier_model = CLIPClassifier().to(
    device
)

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = torch.optim.Adam(

    classifier_model.parameters(),

    lr=1e-4
)

In [ ]:
num_epochs = 10

In [ ]:
start_train_time = time.time()

for epoch in range(num_epochs):

    classifier_model.train()

    total_loss = 0

    for batch in tqdm(train_loader):

        images = batch["image"].to(device)

        texts = batch["text"]

        labels = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = classifier_model(

            images,

            texts
        )

        loss = criterion(

            outputs,

            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = (

        total_loss /

        len(train_loader)
    )

    print(

        f"EPOCH {epoch+1} LOSS:",

        avg_loss
    )

    end_train_time = time.time()

    training_time = (

    end_train_time -

    start_train_time
)

print(

    "TRAINING TIME:",

    training_time
)

In [ ]:
#torch.save(

    #classifier_model.state_dict(),

    #"clip_multioff_model.pth"
#)

In [ ]:
GPU_AVAILABLE = torch.cuda.is_available()

if GPU_AVAILABLE:

    try:

        from pynvml import *

        nvmlInit()

        handle = nvmlDeviceGetHandleByIndex(0)

        ENERGY_MODE = "GPU"

    except:

        ENERGY_MODE = "CPU"

else:

    ENERGY_MODE = "CPU"

In [ ]:
def get_power_usage():

    if ENERGY_MODE == "GPU":

        return (

            nvmlDeviceGetPowerUsage(handle)

            / 1000
        )

    else:

        return 65

In [ ]:
def compute_energy_kwh(

    power_watts,

    duration_seconds
):

    return (

        power_watts *

        duration_seconds

    ) / (1000 * 3600)

In [ ]:
y_true = []

y_pred = []

y_scores = []

inference_times = []

energy_consumptions = []

In [ ]:
classifier_model.eval()

with torch.no_grad():

    for batch in tqdm(test_loader):

        images = batch["image"].to(device)

        texts = batch["text"]

        labels = batch["label"].to(device)

        start_time = time.time()

        power = get_power_usage()

        outputs = classifier_model(

            images,

            texts
        )

        end_time = time.time()

        duration = (

            end_time -

            start_time
        )

        energy = compute_energy_kwh(

            power,

            duration
        )

        probs = torch.softmax(

            outputs,

            dim=1
        )

        preds = torch.argmax(

            probs,

            dim=1
        )

        y_true.extend(

            labels.cpu().numpy()
        )

        y_pred.extend(

            preds.cpu().numpy()
        )

        y_scores.extend(

            probs[:,1].cpu().numpy()
        )

        inference_times.append(
            duration
        )

        energy_consumptions.append(
            energy
        )

In [ ]:
accuracy = accuracy_score(

    y_true,

    y_pred
)

precision = precision_score(

    y_true,

    y_pred
)

recall = recall_score(

    y_true,

    y_pred
)

f1 = f1_score(

    y_true,

    y_pred
)

avg_time = np.mean(
    inference_times
)

avg_energy = np.mean(
    energy_consumptions
)

In [ ]:
print("ACCURACY:", accuracy)

print("PRECISION:", precision)

print("RECALL:", recall)

print("F1-SCORE:", f1)

print("AVERAGE INFERENCE TIME (s):", avg_time)

print("AVERAGE ENERGY (kWh):", avg_energy)

In [ ]:
report = classification_report(

    y_true,

    y_pred,

    target_names=[

        "Non-Offensive",

        "Offensive"
    ],

    digits=4
)

print("\nCLASSIFICATION REPORT:\n")

print(report)

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred
)

labels = [

    "Non-Offensive",

    "Offensive"
]

plt.figure(figsize=(6,5))

plt.imshow(cm)

plt.colorbar()

plt.xticks(
    [0,1],
    labels
)

plt.yticks(
    [0,1],
    labels
)

plt.title("Confusion Matrix")

plt.xlabel("Predicted")

plt.ylabel("Actual")

for i in range(cm.shape[0]):

    for j in range(cm.shape[1]):

        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.show()

In [ ]:
fpr, tpr, thresholds = roc_curve(

    y_true,

    y_scores
)

roc_auc = auc(

    fpr,

    tpr
)

plt.figure(figsize=(6,5))

plt.plot(

    fpr,

    tpr,

    label=f"AUC = {roc_auc:.4f}"
)

plt.plot([0,1],[0,1],'--')

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend()

plt.show()

In [ ]:
results_df = pd.DataFrame({

    "True_Label": y_true,

    "Predicted_Label": y_pred,

    "Prediction_Score": y_scores,

    "Inference_Time": inference_times[:len(y_true)],

    "Energy_kWh": energy_consumptions[:len(y_true)]
})